# Data Loading

In [1]:
import pandas as pd
import re
#import nltk

import spacy

# Import VADER model
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

# Import Doc for extending the SpaCy pipeline
from spacy.tokens import Doc

In [2]:
games_df = pd.read_csv('games.csv')
reviews_df = pd.read_csv('dataset.csv')
#reviews_df.head()

games_df.head()
reviews_df.head()

,app_id,app_name,review_text,review_score,review_votes
0,10,Counter-Strike,Ruined my life.,1,0
1,10,Counter-Strike,This will be more of a ''my experience with th...,1,1
2,10,Counter-Strike,This game saved my virginity.,1,0
3,10,Counter-Strike,• Do you like original games? • Do you like ga...,1,0
4,10,Counter-Strike,"Easy to learn, hard to master.",1,1


In [3]:
listed_genres = set()
for genres in games_df['Genres'].unique().tolist():
    for genre in str(genres).split(','):
        listed_genres.add(genre)


listed_tags = set()
for tags in games_df['Tags'].unique().tolist():
    for tag in str(tags).split(','):
        listed_tags.add(tag)

print(listed_genres)
print(len(listed_genres))

print(listed_tags)
print(len(listed_tags))

{'Action', 'Free To Play', 'Design & Illustration', 'Racing', 'Accounting', 'Violent', 'Photo Editing', '360 Video', 'Casual', 'Strategy', 'Software Training', 'Documentary', 'Movie', 'Tutorial', 'Massively Multiplayer', 'Simulation', 'Web Publishing', 'Utilities', 'Education', 'Short', 'Audio Production', 'Sexual Content', 'Sports', 'Video Production', 'nan', 'Game Development', 'Adventure', 'Early Access', 'Nudity', 'Episodic', 'Animation & Modeling', 'Indie', 'RPG', 'Gore'}
34
{'Mystery', 'Mythology', 'Space', 'Photo Editing', 'Robots', 'Spaceships', 'Politics', 'PvP', 'Free to Play', 'Grid-Based Movement', 'Auto Battler', 'CRPG', 'Character Customization', 'Fast-Paced', 'Rogue-like', 'Tile-Matching', 'Magic', 'Dark Comedy', 'Local Multiplayer', 'Audio Production', 'Match 3', 'Multiple Endings', 'Lovecraftian', '3D Platformer', 'Battle Royale', 'Early Access', 'Gaming', 'Card Battler', 'Satire', 'Real Time Tactics', 'Based On A Novel', 'Tower Defense', 'Racing', 'Pixel Graphics', 'K

In [4]:

narrative_games = games_df[games_df['Tags'].str.contains('Story Rich') | games_df['Tags'].str.contains('Narrative') | games_df['Tags'].str.contains('Choices Matter')]

narrative_games.head()

,AppID,Name,Release date,Estimated owners,Peak CCU,Required age,Price,DiscountDLC count,About the game,Supported languages,...,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Screenshots,Movies
1108640,OMNIMUS,"Sep 25, 2019",0 - 20000,0,0,0.99,0,1,Omnimus is an interactive visual novel in an u...,"['English', 'Russian']",...,0,0,0,Angry games,Total,"Single-player,Family Sharing","Adventure,Indie","Adventure,Indie,Visual Novel,Interactive Ficti...",https://shared.akamai.steamstatic.com/store_it...,NaN
1514380,COGEN: Sword of Rewind / COGEN: 大鳥こはくと刻の剣,"Jan 26, 2022",20000 - 50000,0,0,12.49,0,3,■Defy Death by Rewinding Time! Any hit you tak...,"['English', 'Simplified Chinese', 'Traditional...",...,0,0,0,"Gemdrops, Inc.","Gemdrops, Inc.","Single-player,Steam Achievements,Full controll...",Action,"Action-Adventure,Pixel Graphics,Time Manipulat...",https://shared.akamai.steamstatic.com/store_it...,NaN
1479370,Pixel Game Maker Series Werewolf Princess Kaguya,"Dec 20, 2020",0 - 20000,0,0,9.99,0,0,A stylish &amp; exciting action game! Journey ...,"['English', 'Simplified Chinese', 'Traditional...",...,0,0,0,SERIALGAMES Inc.,Gotcha Gotcha Games,"Single-player,Full controller support,Family S...","Action,Indie","Action,Arcade,Character Action Game,2D Platfor...",https://shared.akamai.steamstatic.com/store_it...,NaN
3098390,Spookstopia: Atomic Division,"Jul 31, 2024",0 - 20000,0,0,2.99,0,0,What is possible between the dimensions of Hel...,"['English', 'Russian']",...,0,0,0,TexFrendza,TexFrendza,"Single-player,Family Sharing","Indie,Early Access","Adventure,Platformer,2D Platformer,2D,Mystery,...",https://shared.akamai.steamstatic.com/store_it...,NaN
3215680,Who is Paul,"Oct 21, 2024",0 - 20000,0,0,0.89,70,0,WARNING: The game's content includes sudden lo...,"['English', 'French', 'Italian', 'German', 'Sp...",...,0,0,0,Atlas Sf,Atlas Sf,"Single-player,Steam Achievements,Family Sharing","Action,Adventure,Indie","Action,Adventure,FPS,Dark,Story Rich,Stealth,A...",https://shared.akamai.steamstatic.com/store_it...,NaN


In [5]:
narrative_game_names = narrative_games["AppID"].unique().tolist()

filtered_reviews = reviews_df[reviews_df["app_name"].isin(narrative_game_names)]

print(len(filtered_reviews))
print(len(filtered_reviews["app_name"].unique().tolist()))

filtered_reviews.head()


1236465
1101


,app_id,app_name,review_text,review_score,review_votes
46177,104600,Portal 2 - The Final Hours,"First of all, this is no game but a virtual in...",1,1
46178,104600,Portal 2 - The Final Hours,The Final Hours is not a game (though can't yo...,1,1
46179,104600,Portal 2 - The Final Hours,Get this if you are a geek about how games are...,1,1
46180,104600,Portal 2 - The Final Hours,IT'S NOT A GAME! But it's a very interesting r...,1,1
46181,104600,Portal 2 - The Final Hours,Everything you would want to know about Portal...,1,1


## Data Cleaning

In [6]:
#filters for ascii characters
filter_ascii = filtered_reviews["review_text"].replace(to_replace=r'[^\x00-\x7f]', value='', regex=True)

#removes new lines
remove_newline = filter_ascii.replace(to_replace=r'\d', value='', regex=True)

#removes certain characters used in reviews
remove_otherchars = remove_newline.replace(to_replace=r'[,()*/"1234567890+|]', value='', regex=True)

#removes punctuations
replace_punctuations = remove_otherchars.replace(to_replace=r'[.!?]+', value = ' | ', regex=True)

#remove dashes not used to hyphenate words
remove_nonhyphen_dashes = replace_punctuations.replace(to_replace=r'- | -', value=' ', regex=True)

#remove extra whitespace
remove_extra_whitespace = remove_nonhyphen_dashes.replace(to_replace=r' +', value=' ', regex=True)

# assign cleaned entries back to the data frame
new = remove_extra_whitespace
filtered_reviews['review_text'] = new

# apply lowercasing
filtered_reviews['review_text'] = filtered_reviews['review_text'].str.lower()
filtered_reviews

print(filtered_reviews)

         app_id                    app_name  \
46177    104600  Portal 2 - The Final Hours   
46178    104600  Portal 2 - The Final Hours   
46179    104600  Portal 2 - The Final Hours   
46180    104600  Portal 2 - The Final Hours   
46181    104600  Portal 2 - The Final Hours   
...         ...                         ...   
6404109    9880            Champions Online   
6404110    9880            Champions Online   
6404111    9880            Champions Online   
6404112    9880            Champions Online   
6404113    9880            Champions Online   

                                               review_text  review_score  \
46177    first of all this is no game but a virtual int...             1   
46178    the final hours is not a game though can't you...             1   
46179    get this if you are a geek about how games are...             1   
46180    it's not a game | but it's a very interesting ...             1   
46181    everything you would want to know about portal

In [7]:
## additional cleaning based on Kaggle User Ali Marjani, PhD
## via https://www.kaggle.com/code/dardodel/steam-reviews-auto-topic-modeling-w-transformers/notebook

print("before removing duplicates: " + str(filtered_reviews.shape))

# drop duplicates
## NOTE: only this actually decreased stuff lmao
filtered_reviews.drop_duplicates(['review_text', 'review_score'], inplace = True)
print("after removing duplicates: " + str(filtered_reviews.shape))


before removing duplicates: (1236465, 5)
after removing duplicates: (1017234, 5)


In [8]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

word_tokens = [ (str(review)).split() for review in filtered_reviews['review_text']]

stop_words = set(stopwords.words('english'))
stop_words.add('us')

lemmatized_reviews = []
for tokens in word_tokens:
    lemmatized_reviews.append([lemmatizer.lemmatize(token) for token in tokens if token not in stop_words or token=='not'])

lemmatized_reviews = [" ".join(review) for review in lemmatized_reviews]

filtered_reviews["lemmatized_reviews"] = lemmatized_reviews
filtered_reviews

,app_id,app_name,review_text,review_score,review_votes,lemmatized_reviews
46177,104600,Portal 2 - The Final Hours,first of all this is no game but a virtual int...,1,1,first game virtual interactive book good one m...
46178,104600,Portal 2 - The Final Hours,the final hours is not a game though can't you...,1,1,final hour not game though can't imagine one s...
46179,104600,Portal 2 - The Final Hours,get this if you are a geek about how games are...,1,1,get geek game made |
46180,104600,Portal 2 - The Final Hours,it's not a game | but it's a very interesting ...,1,1,not game | interesting reading interested crea...
46181,104600,Portal 2 - The Final Hours,everything you would want to know about portal...,1,1,everything would want know portal right intera...
...,...,...,...,...,...,...
6404107,9880,Champions Online,i paid for a month subscription to check out a...,-1,1,paid month subscription check much full game c...
6404108,9880,Champions Online,my absolute favorite superhero games even thou...,-1,0,absolute favorite superhero game even though g...
6404109,9880,Champions Online,the gane starts to get boreing when you reatch...,-1,0,gane start get boreing reatch level know every...
6404110,9880,Champions Online,best character creator i've seen | edit: they ...,-1,0,best character creator seen | edit: removed it...


In [9]:
## Review score of -1: would not recommend

would_recommend = filtered_reviews[filtered_reviews["review_score"]==-1]
print(len(would_recommend))
print(len(would_recommend["app_name"].unique().tolist()))

would_recommend.head()

130621
1029


,app_id,app_name,review_text,review_score,review_votes,lemmatized_reviews
46219,104600,Portal 2 - The Final Hours,i'm sorry to say this glorified pdf file is of...,-1,1,sorry say glorified pdf file little interest a...
46276,104600,Portal 2 - The Final Hours,this game sorry digital book is a rip off | yo...,-1,1,game sorry digital book rip | not paying speci...
46279,104600,Portal 2 - The Final Hours,for those of you who might think this is a gam...,-1,1,might think game not | got little interested d...
46281,104600,Portal 2 - The Final Hours,don't buy this | no seriously this is just an ...,-1,0,buy | seriously article nice package | even vi...
46282,104600,Portal 2 - The Final Hours,unfortunetly every time i go to do something o...,-1,1,unfortunetly every time go something gamebehin...


# Exploratory Data Analysis

In [10]:
#inserting data from 1 dataset to the next
estimated_owners = []
prices = []
last_game = ""
estimate = ""
price = 0
story_rich=[]
narrative=[]
choices_matter=[]
to_append = [False, False, False]
for i in range (len(filtered_reviews)):
    game = filtered_reviews.iloc[i,1] #sometimes its 1, sometimes its 2,,, idk why 
    if game != last_game:
        estimate = narrative_games.loc[narrative_games["AppID"]==game, "Release date"].iloc[0]
        #for some reason realease date is the header for estimated owners dont ask me
        game_tags = narrative_games.loc[narrative_games["AppID"]==game, "Tags"].iloc[0]
        to_append[0] = "Story Rich" in game_tags
        to_append[1] = "Narrative" in game_tags
        to_append[2] = "Choices Matter" in game_tags
        #same here,, required age = price
        price = narrative_games.loc[narrative_games["AppID"]==game, "Required age"].iloc[0]

    estimated_owners.append(estimate)
    story_rich.append(to_append[0])
    narrative.append(to_append[1])
    choices_matter.append(to_append[2])
    prices.append(price)
    last_game = game    


filtered_reviews["estimated_owners"] = estimated_owners 
filtered_reviews["story_rich"] = story_rich
filtered_reviews["narrative"] = narrative 
filtered_reviews["choices_matter"] = choices_matter 
filtered_reviews["price"] = prices

estimated_owners = []
prices = []
last_game = ""
estimate = ""
price = 0
story_rich=[]
narrative=[]
choices_matter=[]
filtered_reviews.head()

,app_id,app_name,review_text,review_score,review_votes,lemmatized_reviews,estimated_owners,story_rich,narrative,choices_matter,price
46177,104600,Portal 2 - The Final Hours,first of all this is no game but a virtual int...,1,1,first game virtual interactive book good one m...,200000 - 500000,True,False,False,0.79
46178,104600,Portal 2 - The Final Hours,the final hours is not a game though can't you...,1,1,final hour not game though can't imagine one s...,200000 - 500000,True,False,False,0.79
46179,104600,Portal 2 - The Final Hours,get this if you are a geek about how games are...,1,1,get geek game made |,200000 - 500000,True,False,False,0.79
46180,104600,Portal 2 - The Final Hours,it's not a game | but it's a very interesting ...,1,1,not game | interesting reading interested crea...,200000 - 500000,True,False,False,0.79
46181,104600,Portal 2 - The Final Hours,everything you would want to know about portal...,1,1,everything would want know portal right intera...,200000 - 500000,True,False,False,0.79


### enable gpu

In [11]:
import torch

# Check if a CUDA-capable GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# This line checks if cuda GPU is available
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.14.1+cu126
True
NVIDIA GeForce RTX 3060 Laptop GPU


# Sentiment Analysis

In [14]:
from IPython.display import clear_output

sentiment_analysis = lambda doc: sia.polarity_scores(doc.text)
nlp_with_sentiment = spacy.load("en_core_web_sm")
Doc.set_extension("sentiment", getter=sentiment_analysis, force=True)

review_docs = nlp_with_sentiment.pipe(lemmatized_reviews, batch_size=32)
i=0
com_sent = []
pos_sent = []
neg_sent = []
neu_sent = []
for i, doc in enumerate(review_docs, start=1):
    sentiment = doc._.sentiment
    com_sent.append(sentiment['compound'])
    pos_sent.append(sentiment['pos'])
    neg_sent.append(sentiment['neg'])
    neu_sent.append(sentiment['neu'])
    i+=1
    if(i%1000==0):
        clear_output(wait=True)
        print(str(i)+'/'+str(len(filtered_reviews)))
filtered_reviews['com_sent'] = com_sent
filtered_reviews['pos_sent'] = pos_sent
filtered_reviews['neg_sent'] = neg_sent
filtered_reviews['neu_sent'] = neu_sent

filtered_reviews.head()

1017000/1017234


,app_id,app_name,review_text,review_score,review_votes,lemmatized_reviews,estimated_owners,story_rich,narrative,choices_matter,price,com_sent,pos_sent,neg_sent,neu_sent
46177,104600,Portal 2 - The Final Hours,first of all this is no game but a virtual int...,1,1,first game virtual interactive book good one m...,200000 - 500000,True,False,False,0.79,0.9047,0.307,0.096,0.597
46178,104600,Portal 2 - The Final Hours,the final hours is not a game though can't you...,1,1,final hour not game though can't imagine one s...,200000 - 500000,True,False,False,0.79,0.0258,0.143,0.112,0.745
46179,104600,Portal 2 - The Final Hours,get this if you are a geek about how games are...,1,1,get geek game made |,200000 - 500000,True,False,False,0.79,-0.2023,0.000,0.310,0.690
46180,104600,Portal 2 - The Final Hours,it's not a game | but it's a very interesting ...,1,1,not game | interesting reading interested crea...,200000 - 500000,True,False,False,0.79,0.9089,0.354,0.082,0.564
46181,104600,Portal 2 - The Final Hours,everything you would want to know about portal...,1,1,everything would want know portal right intera...,200000 - 500000,True,False,False,0.79,0.0772,0.051,0.000,0.949


In [15]:
filtered_reviews.to_csv("output_senti.csv", index=False)

In [12]:
from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer, AutoConfig
from scipy.special import softmax

# from transformers import pipeline
# classifier = pipeline(model='distilbert-base-uncased-finetuned-sst-2-english',task="sentiment-analysis")
SENTIMENT_MODEL = "distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(SENTIMENT_MODEL)
config = AutoConfig.from_pretrained(SENTIMENT_MODEL)

model = AutoModelForSequenceClassification.from_pretrained(SENTIMENT_MODEL)
model.save_pretrained(SENTIMENT_MODEL)

model.to(device)
model.eval()


d:\4th year\1st sem\soc comp\final project\projectvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.01it/s]


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout(

In [13]:
all_texts = []
review_positions = []

for row_pos, review in enumerate(filtered_reviews['lemmatized_reviews'].tolist()):

    if not isinstance(review, str):
        continue

    for text in review.split('|'):
        text = text.strip()

        all_texts.append(text)
        review_positions.append(row_pos)

In [ ]:
import numpy as np
BATCH_SIZE = 32


filtered_reviews['pos_sent'] = 0.0
filtered_reviews['neg_sent'] = 0.0



i=0
for index, row in filtered_reviews.iterrows():
    texts = row['lemmatized_reviews']
    temp_neg = []
    temp_neu = []
    temp_pos = []
    for text in texts.split('|'):
        if text.isspace() or len(text)==0:
            continue
        encoded_input = tokenizer(text, return_tensors='pt', truncation=True, max_length=512)
        encoded_input = {
            key: value.to(device)
            for key, value in encoded_input.items()
        }
        with torch.no_grad():
            output = model(**encoded_input)
        
        scores = output[0][0].cpu().numpy()
        scores = softmax(scores)
        temp_neg.append(scores[0])
        temp_pos.append(scores[1])

    if(len(temp_neg)==0):
        temp_neg.append(0)
        temp_pos.append(0)
    
    filtered_reviews.at[index, 'pos_sent'] = sum(temp_pos)/len(temp_pos)
    filtered_reviews.at[index, 'neg_sent'] = sum(temp_neg)/len(temp_neg)

    i+=1
    if(i==1028):
        break

filtered_reviews.head()


,app_id,app_name,review_text,review_score,review_votes,lemmatized_reviews,estimated_owners,story_rich,narrative,choices_matter,price,pos_sent,neg_sent,neu_sent
46177,104600.0,Portal 2 - The Final Hours,first of all this is no game but a virtual int...,1.0,1.0,first game virtual interactive book good one m...,200000 - 500000,True,False,False,0.79,0.985086,0.014914,0.0
46178,104600.0,Portal 2 - The Final Hours,the final hours is not a game though can't you...,1.0,1.0,final hour not game though can't imagine one s...,200000 - 500000,True,False,False,0.79,0.406472,0.593528,0.0
46179,104600.0,Portal 2 - The Final Hours,get this if you are a geek about how games are...,1.0,1.0,get geek game made |,200000 - 500000,True,False,False,0.79,0.991366,0.008634,0.0
46180,104600.0,Portal 2 - The Final Hours,it's not a game | but it's a very interesting ...,1.0,1.0,not game | interesting reading interested crea...,200000 - 500000,True,False,False,0.79,0.740863,0.259137,0.0
46181,104600.0,Portal 2 - The Final Hours,everything you would want to know about portal...,1.0,1.0,everything would want know portal right intera...,200000 - 500000,True,False,False,0.79,0.903454,0.096546,0.0


# Topic Modeling

## Auto Topic Modeling with Transofmers
Based on the methodology of Kaggle User Ali Marjani, PhD in the following notebook: https://www.kaggle.com/code/dardodel/steam-reviews-auto-topic-modeling-w-transformers/notebook

In [ ]:
import numpy as np 
import pandas as pd 
import random as rn
import re
import nltk
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 

pd.set_option("display.max_rows", 600)
pd.set_option("display.max_columns", 500)
pd.set_option("max_colwidth", 400)

In [ ]:
willrecco_reviews = filtered_reviews[filtered_reviews.review_score==1]
print(len(willrecco_reviews))

## General Topic Modeling

In [ ]:
#General topic modeling
from gensim.corpora.dictionary import Dictionary
from gensim.models.ldamodel import LdaModel
import pyLDAvis, pyLDAvis.gensim_models

In [ ]:
recco_texts = []
recco_texts_sentences = []

notrecco_texts = []
notrecco_texts_sentences = []

###

recco = filtered_reviews[filtered_reviews["review_score"]==1]
recco_lemmatized = [review for review in recco["lemmatized_reviews"]]
print(len(recco_lemmatized))

for i in range(len(recco_lemmatized)):
    for sentence in recco_lemmatized[i].split(' | '):
        if len(sentence.split()) > 5:
            recco_texts_sentences.append([word for word in sentence.split() if word.isalnum() and word not in stop_words])
    if len((re.sub(r' \| ',' ', recco_lemmatized[i])).split())>5:
        recco_texts.append([word for word in (re.sub(r' \| ',' ', recco_lemmatized[i])).split() if word.isalnum() and word not in stop_words])

notrecco = filtered_reviews[filtered_reviews["review_score"]==-1]
notrecco_lemmatized = [review for review in notrecco["lemmatized_reviews"]]
print(len(notrecco_lemmatized))

for i in range(len(notrecco_lemmatized)):
    for sentence in notrecco_lemmatized[i].split(' | '):
        if len(sentence.split()) > 5:
            notrecco_texts_sentences.append([word for word in sentence.split() if word.isalnum() and word not in stop_words])
    if len((re.sub(r' \| ',' ', notrecco_lemmatized[i])).split())>5:
        notrecco_texts.append([word for word in (re.sub(r' \| ',' ', notrecco_lemmatized[i])).split() if word.isalnum() and word not in stop_words])

In [ ]:
print("RECCO TEXTS LEN:     " + str(len(recco_texts)))
print("RECCO SENTS LEN:     " + str(len(recco_texts_sentences)))
print("NOTRECCO TEXTS LEN:  " + str(len(notrecco_texts)))
print("NOTRECCO SENTS LEN:  " + str(len(notrecco_texts_sentences)))

In [ ]:
dictionary = Dictionary(recco_texts + notrecco_texts) # Create a GenSimdictionary
dictionary.filter_extremes(no_below=20, no_above=0.8) #word has to appear in 20 documents,, idk if thats fair
corpus = [dictionary.doc2bow(text) for text in (recco_texts + notrecco_texts)] # Create a GenSim corpus
lda = LdaModel(
    corpus=corpus, 
    id2word=dictionary, 
    num_topics=7
)

pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim_models.prepare(lda, corpus, dictionary,mds='mmds')

pyLDAvis.display(vis)

In [ ]:
dictionary2 = Dictionary(recco_texts_sentences + notrecco_texts_sentences) # Create a GenSimdictionary
dictionary2.filter_extremes(no_below=20, no_above=0.8) #word has to appear in 20 documents,, idk if thats fair
corpus2 = [dictionary2.doc2bow(text) for text in (recco_texts_sentences + notrecco_texts_sentences)] # Create a GenSim corpus
lda2 = LdaModel(
    corpus=corpus2, 
    id2word=dictionary2, 
    num_topics=7
)
    
pyLDAvis.enable_notebook()
vis2 = pyLDAvis.gensim_models.prepare(lda2, corpus2, dictionary2,mds='mmds')

pyLDAvis.display(vis2)

### Topic Modeling Across Recommended / Not Recommended

In [ ]:
dictionaryReccoTexts = Dictionary(recco_texts) # Create a GenSimdictionary
dictionaryReccoTexts.filter_extremes(no_below=20, no_above=0.8) #word has to appear in 20 documents,, idk if thats fair
corpusRT = [dictionaryReccoTexts.doc2bow(text) for text in recco_texts] # Create a GenSim corpus
ldaRT = LdaModel(
    corpus=corpusRT, 
    id2word=dictionaryReccoTexts, 
    num_topics=7
)
    
pyLDAvis.enable_notebook()
visRT = pyLDAvis.gensim_models.prepare(ldaRT, corpusRT, dictionaryReccoTexts,mds='mmds')

pyLDAvis.display(visRT)

In [ ]:
dictionaryReccoSentences = Dictionary(recco_texts_sentences) # Create a GenSimdictionary
dictionaryReccoSentences.filter_extremes(no_below=20, no_above=0.8) #word has to appear in 20 documents,, idk if thats fair
corpusRS = [dictionaryReccoSentences.doc2bow(text) for text in recco_texts_sentences] # Create a GenSim corpus
ldaRS = LdaModel(
    corpus=corpusRS, 
    id2word=dictionaryReccoSentences, 
    num_topics=7
)
    
pyLDAvis.enable_notebook()
visRS = pyLDAvis.gensim_models.prepare(ldaRS, corpusRS, dictionaryReccoSentences,mds='mmds')

pyLDAvis.display(visRS)

In [ ]:
dictionaryNotReccoTexts = Dictionary(notrecco_texts) # Create a GenSimdictionary
dictionaryNotReccoTexts.filter_extremes(no_below=20, no_above=0.8) #word has to appear in 20 documents,, idk if thats fair
corpusNRT = [dictionaryNotReccoTexts.doc2bow(text) for text in notrecco_texts] # Create a GenSim corpus
ldaNRT = LdaModel(
    corpus=corpusNRT, 
    id2word=dictionaryNotReccoTexts, 
    num_topics=7
)
    
pyLDAvis.enable_notebook()
visNRT = pyLDAvis.gensim_models.prepare(ldaNRT, corpusNRT, dictionaryNotReccoTexts,mds='mmds')

pyLDAvis.display(visNRT)